# Kaggle GPU Training — Existing From-Scratch Medical AI Models (EN/FR Conversations)

Trains the **existing** `HybridPipeline`, `ClassicalCRFModel`, and `TorchCharCNNBiLSTMCRF` from scratch (0 pretrained weights) on `medical_conversation_dataset_en_fr.zip` + `SYNTH-MED-V3.1`, saving versioned checkpoints to `/kaggle/working/models_out` for Intel CPU inference.

In [ ]:
import os, subprocess, sys, torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0), '| VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))


In [ ]:
# Install only minimal required dependencies (no transformers, no timm, no pretrained downloads)
!pip install -q --no-deps sklearn-crfsuite python-crfsuite pyyaml
!pip install -q -e . --no-deps


In [ ]:
# Verify dataset archive integrity and zero split leakage before training
from pathlib import Path
from medresearch_platform.data.conversation_loader import load_conversation_dataset_from_zip

ZIP_PATH = Path('/kaggle/input/medical-conversation-dataset-en-fr/medical_conversation_dataset_en_fr.zip')
if not ZIP_PATH.exists():
    ZIP_PATH = Path('medical_conversation_dataset_en_fr.zip')
ds = load_conversation_dataset_from_zip(ZIP_PATH)
print('Dataset audit:', ds.audit)


In [ ]:
# Train existing models (HybridPipeline + ClassicalCRF + PyTorch Char-CNN-BiLSTM-CRF) and save to persistent Kaggle output
OUT_DIR = Path('/kaggle/working/models_out') if Path('/kaggle/working').exists() else Path('models')
REPORT_OUT = OUT_DIR / 'conversation_post_training_eval.json'
cmd = [
    sys.executable, 'scripts/train_conversation_models.py',
    '--zip-path', str(ZIP_PATH),
    '--out-dir', str(OUT_DIR),
    '--report-out', str(REPORT_OUT),
    '--seed', '1',
    '--train-torch',
    '--torch-epochs', '16',
    '--torch-batch-size', '16',
    '--torch-lr', '0.001',
]
subprocess.run(cmd, check=True)
